# LeetCode 3445 — Maximum Difference Between Even and Odd Frequency II bằng Python

- Link: https://leetcode.com/problems/maximum-difference-between-even-and-odd-frequency-ii/
- Độ khó: **Hard**
- Chủ đề: **Cố định cặp ký tự** `(a, b)`, **tổng tiền tố**, **trạng thái chẵn lẻ** 2 bit, **hai con trỏ** (cửa sổ trượt) để giữ điều kiện độ dài và "b xuất hiện"

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def maxDifference(self, s: str, k: int) -> int:
```

## 1. Mục tiêu học

1. Bảng chữ cái chỉ 5 ký tự ⇒ **cố định cặp `(a, b)`** (20 cặp) để đơn giản bài toán.
2. Viết `freq_a − freq_b` của xâu con `(l, r]` thành **hiệu tiền tố** `D[r] − D[l]` với `D = P_a − P_b`.
3. Mã hóa điều kiện "lẻ/chẵn" bằng **tính chẵn lẻ của tiền tố**: cần `(P_a[l] mod 2, P_b[l] mod 2) = (1 − P_a[r] mod 2, P_b[r] mod 2)`.
4. Hai ràng buộc "độ dài ≥ k" và "`b` xuất hiện ít nhất một lần" đều **đơn điệu theo `r`** ⇒ con trỏ `l` chỉ tiến.
5. Giữ `min D[l]` cho từng trong 4 trạng thái chẵn lẻ ⇒ `O(20 n)`.

## 2. Đề bài & ví dụ

Cho `s` (chỉ gồm `'0'..'4'`) và `k`. Tìm giá trị lớn nhất của `freq[a] − freq[b]` trên một xâu con `subs` sao cho: `|subs| ≥ k`, `freq[a]` **lẻ**, `freq[b]` **chẵn và khác 0**.

**Ví dụ 1:** `"12233", k = 4` → `-1`. **Ví dụ 2:** `"1122211", k = 3` → `1`. **Ví dụ 3:** `"110", k = 3` → `-1`.

**Ràng buộc:** `3 ≤ n ≤ 3·10^4`; đảm bảo có ít nhất một xâu con hợp lệ.

In [1]:
examples = [("12233", 4, -1), ("1122211", 3, 1), ("110", 3, -1)]
for s, k, exp in examples:
    print(s, k, '-> kỳ vọng', exp)

12233 4 -> kỳ vọng -1
1122211 3 -> kỳ vọng 1
110 3 -> kỳ vọng -1


## 3. Kiến thức nền

### 3.1. Tổng tiền tố đếm ký tự

`P_c[i]` = số lần ký tự `c` xuất hiện trong `s[0..i)`. Xâu con `s[l..r)` có `freq[c] = P_c[r] − P_c[l]`.

In [2]:
def prefix(s, c):
    P = [0]
    for ch in s:
        P.append(P[-1] + (ch == c))
    return P

print(prefix("1122211", '1'), prefix("1122211", '2'))

[0, 1, 2, 2, 2, 2, 3, 4] [0, 0, 0, 1, 2, 3, 3, 3]


### 3.2. Tính chẵn lẻ của hiệu

`P[r] − P[l]` lẻ ⇔ `P[r]` và `P[l]` **khác** tính chẵn lẻ; chẵn ⇔ **cùng** tính chẵn lẻ. Mã hóa trạng thái tiền tố bằng 2 bit: `st(l) = 2·(P_a[l] mod 2) + (P_b[l] mod 2)`.

### 3.3. Giữ min theo nhóm khi con trỏ trái tiến

Nếu tập các `l` hợp lệ chỉ **mở rộng** khi `r` tăng, ta chỉ cần thêm dần `l` vào và giữ `min` cho mỗi nhóm trạng thái — không bao giờ phải xóa.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 2 `"1122211"`, `k = 3`, chọn `a = '2'`, `b = '1'`: xâu con `"11222"` có `freq[2] = 3` (lẻ), `freq[1] = 2` (chẵn, ≠ 0) ⇒ hiệu `1`.

Với tiền tố: `P_2 = [0,0,0,1,2,3,3,3]`, `P_1 = [0,1,2,2,2,2,3,4]`. Xâu con `s[0..5)`: `l = 0`, `r = 5`: `P_2[5] − P_2[0] = 3` lẻ, `P_1[5] − P_1[0] = 2` chẵn ⇒ `D[5] − D[0] = (3 − 2) − (0 − 0) = 1`.

In [3]:
s = "1122211"
P2, P1 = prefix(s, '2'), prefix(s, '1')
D = [x - y for x, y in zip(P2, P1)]
print('D =', D, '| D[5] - D[0] =', D[5] - D[0])

D = [0, -1, -2, -1, 0, 1, 0, -1] | D[5] - D[0] = 1


## 5. Cách 1 — Brute Force: mọi xâu con, mọi cặp ký tự

In [4]:
NEG = float('-inf')

def max_diff_brute(s, k):
    n, best = len(s), NEG
    for i in range(n):
        cnt = [0] * 5
        for j in range(i, n):
            cnt[ord(s[j]) - 48] += 1
            if j - i + 1 >= k:
                for a in range(5):
                    if cnt[a] % 2 == 1:
                        for b in range(5):
                            if cnt[b] and cnt[b] % 2 == 0:
                                best = max(best, cnt[a] - cnt[b])
    return best

for s, k, exp in examples:
    print(max_diff_brute(s, k), '| kỳ vọng', exp)

-1 | kỳ vọng -1
1 | kỳ vọng 1
-1 | kỳ vọng -1


**Độ phức tạp:** `O(n² · 25)` ≈ `2·10^{10}` với `n = 3·10^4`: quá chậm.

## 6. Ý tưởng thuật toán / bước đột phá

Cố định `(a, b)`, `D[i] = P_a[i] − P_b[i]`. Với mỗi `r` (đầu phải, loại trừ), cần `l` thỏa:

1. `r − l ≥ k` ⇔ `l ≤ r − k`;
2. `freq[b] > 0` ⇔ `P_b[l] < P_b[r]`;
3. tính chẵn lẻ: `st(l) = 2·(1 − P_a[r] mod 2) + (P_b[r] mod 2)`.

Tối đa hóa `D[r] − D[l]` ⇔ `min D[l]` trong nhóm trạng thái phù hợp. Điều kiện 1 và 2 đều **chỉ nới lỏng** khi `r` tăng (`r − k` tăng, `P_b[r]` không giảm) và với `l` tăng thì khó thỏa hơn ⇒ tập `l` hợp lệ là một **tiền tố** `[0, L(r))` với `L(r)` không giảm ⇒ hai con trỏ.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Cố định cặp, duyệt mọi `l` (O(20 n²))

Bước trung gian: dùng tiền tố để kiểm tra điều kiện trong `O(1)` thay vì đếm lại.

In [5]:
def max_diff_pairs_quadratic(s, k):
    n, ans = len(s), NEG
    for a in '01234':
        for b in '01234':
            if a == b:
                continue
            Pa, Pb = prefix(s, a), prefix(s, b)
            for r in range(k, n + 1):
                for l in range(0, r - k + 1):
                    fa, fb = Pa[r] - Pa[l], Pb[r] - Pb[l]
                    if fa % 2 == 1 and fb > 0 and fb % 2 == 0:
                        ans = max(ans, fa - fb)
    return ans

for s, k, exp in examples:
    print(max_diff_pairs_quadratic(s, k), '| kỳ vọng', exp)

-1 | kỳ vọng -1
1 | kỳ vọng 1
-1 | kỳ vọng -1


### 7.2. Cách 3 — Cố định cặp + trạng thái chẵn lẻ + hai con trỏ (lời giải chính)

In [6]:
INF = float('inf')

def max_diff_two_ptr(s, k):
    n, ans = len(s), NEG
    for a in '01234':
        for b in '01234':
            if a == b:
                continue
            Pa, Pb = prefix(s, a), prefix(s, b)
            best = [INF] * 4
            l = 0
            for r in range(k, n + 1):
                while l <= r - k and Pb[l] < Pb[r]:
                    st = (Pa[l] & 1) << 1 | (Pb[l] & 1)
                    v = Pa[l] - Pb[l]
                    if v < best[st]:
                        best[st] = v
                    l += 1
                need = ((Pa[r] & 1) ^ 1) << 1 | (Pb[r] & 1)
                if best[need] < INF:
                    ans = max(ans, Pa[r] - Pb[r] - best[need])
    return ans

for s, k, exp in examples:
    print(max_diff_two_ptr(s, k), '| kỳ vọng', exp)

-1 | kỳ vọng -1
1 | kỳ vọng 1
-1 | kỳ vọng -1


### 7.3. Cách 4 — Bỏ qua cặp `(a, b)` không xuất hiện + vòng lặp trên mảng ký tự đã mã hóa

Tối ưu hằng số: mã hóa `s` thành danh sách số, chỉ xét cặp mà cả `a` và `b` đều có trong `s`, và tính tiền tố bằng một vòng lặp duy nhất cho cặp đó. Cùng thuật toán với cách 3.

In [7]:
def max_diff_fast(s, k):
    arr = [ord(c) - 48 for c in s]
    present = set(arr)
    n, ans = len(arr), NEG
    for a in present:
        for b in present:
            if a == b:
                continue
            Pa = [0] * (n + 1); Pb = [0] * (n + 1)
            ca = cb = 0
            for i, x in enumerate(arr):
                if x == a: ca += 1
                elif x == b: cb += 1
                Pa[i + 1] = ca; Pb[i + 1] = cb
            best = [INF] * 4
            l = 0
            for r in range(k, n + 1):
                pbr = Pb[r]
                while l <= r - k and Pb[l] < pbr:
                    st = (Pa[l] & 1) << 1 | (Pb[l] & 1)
                    v = Pa[l] - Pb[l]
                    if v < best[st]:
                        best[st] = v
                    l += 1
                bv = best[((Pa[r] & 1) ^ 1) << 1 | (pbr & 1)]
                if bv < INF and Pa[r] - pbr - bv > ans:
                    ans = Pa[r] - pbr - bv
    return ans

for s, k, exp in examples:
    print(max_diff_fast(s, k), '| kỳ vọng', exp)

-1 | kỳ vọng -1
1 | kỳ vọng 1
-1 | kỳ vọng -1


## 8. TRACE từng bước

Trace cách 3 trên ví dụ 2 với cặp `a = '2'`, `b = '1'`.

In [8]:
def trace_pair(s, k, a, b):
    n = len(s)
    Pa, Pb = prefix(s, a), prefix(s, b)
    best = [INF] * 4
    l, ans = 0, NEG
    print('Pa =', Pa, '| Pb =', Pb)
    for r in range(k, n + 1):
        added = []
        while l <= r - k and Pb[l] < Pb[r]:
            st = (Pa[l] & 1) << 1 | (Pb[l] & 1)
            best[st] = min(best[st], Pa[l] - Pb[l])
            added.append(l)
            l += 1
        need = ((Pa[r] & 1) ^ 1) << 1 | (Pb[r] & 1)
        cand = Pa[r] - Pb[r] - best[need] if best[need] < INF else None
        if cand is not None:
            ans = max(ans, cand)
        print(f"r={r}: thêm l={added}, best={best}, cần trạng thái {need:02b} -> ứng viên {cand}")
    print('tốt nhất cho cặp', (a, b), ':', ans)

trace_pair("1122211", 3, '2', '1')

Pa = [0, 0, 0, 1, 2, 3, 3, 3] | Pb = [0, 1, 2, 2, 2, 2, 3, 4]
r=3: thêm l=[0], best=[0, inf, inf, inf], cần trạng thái 00 -> ứng viên -1
r=4: thêm l=[1], best=[0, -1, inf, inf], cần trạng thái 10 -> ứng viên None
r=5: thêm l=[], best=[0, -1, inf, inf], cần trạng thái 00 -> ứng viên 1
r=6: thêm l=[2, 3], best=[-2, -1, -1, inf], cần trạng thái 01 -> ứng viên 1
r=7: thêm l=[4], best=[-2, -1, -1, inf], cần trạng thái 00 -> ứng viên 1
tốt nhất cho cặp ('2', '1') : 1


## 9. Code nộp LeetCode

In [9]:
class Solution:
    def maxDifference(self, s: str, k: int) -> int:
        arr = [ord(c) - 48 for c in s]
        n = len(arr)
        present = set(arr)
        INF = float('inf')
        ans = -INF
        for a in present:
            for b in present:
                if a == b:
                    continue
                Pa = [0] * (n + 1)
                Pb = [0] * (n + 1)
                ca = cb = 0
                for i, x in enumerate(arr):
                    if x == a:
                        ca += 1
                    elif x == b:
                        cb += 1
                    Pa[i + 1] = ca
                    Pb[i + 1] = cb
                best = [INF] * 4
                l = 0
                for r in range(k, n + 1):
                    while l <= r - k and Pb[l] < Pb[r]:
                        st = (Pa[l] & 1) << 1 | (Pb[l] & 1)
                        v = Pa[l] - Pb[l]
                        if v < best[st]:
                            best[st] = v
                        l += 1
                    bv = best[((Pa[r] & 1) ^ 1) << 1 | (Pb[r] & 1)]
                    if bv < INF and Pa[r] - Pb[r] - bv > ans:
                        ans = Pa[r] - Pb[r] - bv
        return int(ans)

for s, k, exp in examples:
    got = Solution().maxDifference(s, k)
    print(got, 'OK' if got == exp else 'SAI')

-1 OK
1 OK
-1 OK


## 10. Vì sao thuật toán đúng

1. **Phủ hết:** mọi xâu con hợp lệ có một cặp `(a, b)` (`a ≠ b`, vì một tần số lẻ, một chẵn ≠ 0) — xét đủ 20 cặp.
2. **Biến đổi:** `freq_a − freq_b = D[r] − D[l]`; điều kiện lẻ/chẵn tương đương điều kiện trạng thái `st(l)` (mục 3.2); `freq_b > 0` ⇔ `P_b[l] < P_b[r]`; độ dài ⇔ `l ≤ r − k`.
3. **Tập `l` hợp lệ (bỏ qua chẵn lẻ) là tiền tố `[0, L(r))`:** vì `P_b` không giảm, nếu `l` thỏa `P_b[l] < P_b[r]` thì mọi `l' < l` cũng thỏa; tương tự `l ≤ r − k`. Khi `r` tăng, `r − k` tăng và `P_b[r]` không giảm ⇒ `L(r)` không giảm ⇒ con trỏ chỉ tiến, mỗi `l` thêm đúng một lần.
4. **`best[st]`** = `min D[l]` trên các `l` đã thêm có trạng thái `st` ⇒ ứng viên tốt nhất cho `r` đúng là `D[r] − best[need]`.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Mọi xâu con | `O(25 n²)` | `O(1)` |
| 2. Cặp + mọi `l` | `O(20 n²)` | `O(n)` |
| 3. Cặp + hai con trỏ | `O(20 n)` | `O(n)` |
| 4. Như 3, tối ưu hằng số | `O(20 n)` | `O(n)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [10]:
impls = {'quadratic': max_diff_pairs_quadratic, 'two_ptr': max_diff_two_ptr, 'fast': max_diff_fast,
         'submit': lambda s, k: Solution().maxDifference(s, k)}
edge = [("011", 3, -1), ("0111", 1, -1), ("00011", 5, 1), ("11000", 1, 1), ("0101010", 4, 1)]
for s, k, exp in examples + edge:
    assert max_diff_brute(s, k) == exp, (s, k, max_diff_brute(s, k))
    for name, f in impls.items():
        assert f(s, k) == exp, (name, s, k)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với vét cạn

In [11]:
import random
rng = random.Random(3445)
cnt = 0
for _ in range(2500):
    n = rng.randint(3, 16)
    alpha = '01234'[:rng.randint(2, 5)]
    s = ''.join(rng.choice(alpha) for _ in range(n))
    k = rng.randint(1, n)
    exp = max_diff_brute(s, k)
    if exp == NEG:
        continue                               # đề đảm bảo luôn có xâu con hợp lệ
    for name, f in impls.items():
        assert f(s, k) == exp, (name, s, k)
    cnt += 1
print('Khớp vét cạn trên', cnt, 'bộ ngẫu nhiên hợp lệ.')

Khớp vét cạn trên 2325 bộ ngẫu nhiên hợp lệ.


### 12.3. Ca lớn `n = 3·10^4`

In [12]:
import time
n = 3 * 10 ** 4
big = {
    'ngẫu nhiên, k = 5': (''.join(rng.choice('01234') for _ in range(n)), 5),
    'ngẫu nhiên, k = n/2': (''.join(rng.choice('01234') for _ in range(n)), n // 2),
    'hai ký tự': (''.join(rng.choice('01') for _ in range(n)), 100),
    'khối dài': ('0' * (n // 2) + '1' * (n // 2 - 1) + '2', 3),
}
for name, (s, k) in big.items():
    t = time.time(); a = Solution().maxDifference(s, k); t1 = time.time() - t
    t = time.time(); b = max_diff_two_ptr(s, k); t2 = time.time() - t
    assert a == b
    print(f"{name:20s}: {a:6d} | nộp bài {t1:.2f}s | cách 3 {t2:.2f}s")

ngẫu nhiên, k = 5   :    295 | nộp bài 0.20s | cách 3 0.23s


ngẫu nhiên, k = n/2 :    207 | nộp bài 0.12s | cách 3 0.12s
hai ký tự           :    361 | nộp bài 0.02s | cách 3 0.14s


khối dài            :  14997 | nộp bài 0.05s | cách 3 0.12s


## 13. Các lỗi người mới hay gặp

Hàm tìm phản ví dụ (seed cố định):

In [13]:
def find_counterexample(wrong, seed=0, tries=20000):
    rng = random.Random(seed)
    best = None
    for _ in range(tries):
        n = rng.randint(3, 8)
        s = ''.join(rng.choice('012') for _ in range(n))
        k = rng.randint(1, n)
        exp = max_diff_brute(s, k)
        if exp == NEG:
            continue
        if wrong(s, k) != exp and (best is None or len(s) < len(best[0])):
            best = (s, k)
    return best

def make_solver(require_b_present=True, flip_a=True, use_k=True, init=NEG):
    def solve(s, k):
        n, ans = len(s), init
        for a in '01234':
            for b in '01234':
                if a == b:
                    continue
                Pa, Pb = prefix(s, a), prefix(s, b)
                best = [INF] * 4
                l = 0
                for r in range(1, n + 1):
                    lim = r - k if use_k else r - 1
                    while l <= lim and (not require_b_present or Pb[l] < Pb[r]):
                        st = (Pa[l] & 1) << 1 | (Pb[l] & 1)
                        best[st] = min(best[st], Pa[l] - Pb[l])
                        l += 1
                    pa_bit = (Pa[r] & 1) ^ 1 if flip_a else (Pa[r] & 1)
                    need = pa_bit << 1 | (Pb[r] & 1)
                    if best[need] < INF:
                        ans = max(ans, Pa[r] - Pb[r] - best[need])
        return ans
    return solve

### Lỗi 1 — Cho phép `freq[b] = 0` (0 cũng là số chẵn)

In [14]:
wrong_zero_b = make_solver(require_b_present=False)
ce = find_counterexample(wrong_zero_b, seed=1)
print('phản ví dụ:', ce, '| sai:', wrong_zero_b(*ce), '| đúng:', max_diff_brute(*ce))

phản ví dụ: ('220', 2) | sai: 1 | đúng: -1


**Vì sao sai:** đề yêu cầu tần số chẵn **khác 0**. Bỏ điều kiện này cho phép "trừ 0" và làm đáp án lớn hơn thực tế.

### Lỗi 2 — Dùng cùng tính chẵn lẻ cho `a` (quên rằng hiệu lẻ cần khác tính chẵn lẻ)

In [15]:
wrong_parity = make_solver(flip_a=False)
ce = find_counterexample(wrong_parity, seed=2)
print('phản ví dụ:', ce, '| sai:', wrong_parity(*ce), '| đúng:', max_diff_brute(*ce))

phản ví dụ: ('001', 1) | sai: -2 | đúng: -1


**Vì sao sai:** `P_a[r] − P_a[l]` lẻ ⇔ hai tiền tố **khác** tính chẵn lẻ.

### Lỗi 3 — Quên ràng buộc độ dài `≥ k`

In [16]:
wrong_len = make_solver(use_k=False)
ce = find_counterexample(wrong_len, seed=3)
print('phản ví dụ:', ce, '| sai:', wrong_len(*ce), '| đúng:', max_diff_brute(*ce))

phản ví dụ: ('01111', 5) | sai: -1 | đúng: -3


**Vì sao sai:** xâu con ngắn hơn `k` không được chọn.

### Lỗi 4 — Khởi tạo đáp án bằng `0`

In [17]:
wrong_init = make_solver(init=0)
print('sai :', wrong_init("12233", 4), '| đúng:', Solution().maxDifference("12233", 4))

sai : 0 | đúng: -1


**Vì sao sai:** đáp án có thể **âm** (ví dụ 1, 3). Khởi tạo bằng `−∞`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Bảng chữ cái nhỏ ⇒ cố định cặp/tập ký tự rồi giải bài con đơn giản.
- "Tần số lẻ/chẵn trong xâu con" ⇒ trạng thái chẵn lẻ của **tiền tố** (bitmask).
- "Tối đa hiệu tiền tố với ràng buộc trên `l`" ⇒ giữ `min` theo nhóm, con trỏ trái tiến đơn điệu.
- Điều kiện "xuất hiện ít nhất một lần" ⇒ `P_b[l] < P_b[r]`, cũng đơn điệu.

## 15. Biến thể của bài

1. **LC 3442 — bản I:** xét cả xâu (không cần xâu con) ⇒ đếm tần số rồi lấy max lẻ − min chẵn.
2. Bỏ điều kiện "khác 0" ⇒ bỏ ràng buộc `P_b[l] < P_b[r]`.
3. Tối đa `freq[a] + freq[b]` với `a` lẻ, `b` chẵn ⇒ đổi dấu `D = P_a + P_b`.
4. LC 1371 (Longest Substring Containing Vowels in Even Counts) — trạng thái chẵn lẻ tiền tố.

In [18]:
# Biến thể 1 (LC 3442): toàn bộ xâu
from collections import Counter

def max_diff_whole(s):
    c = Counter(s)
    odd = [v for v in c.values() if v % 2 == 1]
    even = [v for v in c.values() if v % 2 == 0]
    return max(odd) - min(even)

assert max_diff_whole("aaaaabbc") == 3 and max_diff_whole("abcabcab") == 1      # ví dụ LC 3442
# Liên hệ: bản II với k = n chính là bản I (trên bảng chữ cái 0..4)
rng = random.Random(15)
for _ in range(300):
    s = ''.join(rng.choice('0123') for _ in range(rng.randint(3, 15)))
    c = Counter(s)
    if any(v % 2 for v in c.values()) and any(v % 2 == 0 for v in c.values()):
        assert Solution().maxDifference(s, len(s)) == max_diff_whole(s)
print('Bản II với k = n trùng bản I trên 300 bộ.')

Bản II với k = n trùng bản I trên 300 bộ.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Mọi xâu con | `O(25 n²)` | Oracle |
| 2 | Cặp + mọi `l` | `O(20 n²)` | Trung gian |
| 3 | Cặp + chẵn lẻ + hai con trỏ | `O(20 n)` | **Ý tưởng chính** |
| 4 | Như 3, tối ưu hằng số | `O(20 n)` | **Nộp bài** |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `s = "0011100"`, `k = 7`.

**Bài 2 (tính tay):** `s = "01110"`, `k = 2`.

**Bài 3 (điền TODO):** viết `max_diff_allow_zero(s, k)` — biến thể 2 (cho phép `freq[b] = 0`).

**Bài 4 (tự viết lại):** trả về cả **vị trí** `(l, r)` của một xâu con tối ưu.

**Bài 5 (mini challenge):** biến thể 3 — tối đa `freq[a] + freq[b]` (`a` lẻ, `b` chẵn khác 0, độ dài ≥ k).

In [19]:
def max_diff_allow_zero_todo(s, k):
    # TODO
    raise NotImplementedError

def best_substring_todo(s, k):
    # TODO
    raise NotImplementedError

def max_sum_odd_even_todo(s, k):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `k = n` ⇒ cả xâu. (2) `freq[0] = 4`, `freq[1] = 3`. (3) `3 − 4 = −1`.

**Bài 2.** (1) Thử `"111"`: `freq[1] = 3` nhưng không có ký tự chẵn ≠ 0. (2) `"0110"`? `freq[0] = 2`, `freq[1] = 2` — không có lẻ. (3) `"01110"`: `3 − 2 = 1`.

**Bài 3.** (1) Bỏ `P_b[l] < P_b[r]`. (2) Vẫn giữ `l ≤ r − k`. (3) Đối chiếu vét cạn có cho phép `cnt[b] = 0`.

**Bài 4.** (1) Lưu thêm chỉ số `l` đạt `min` trong `best[st]`. (2) Khi cập nhật đáp án, lưu `(l, r)`. (3) Kiểm tra lại bằng đếm trực tiếp.

**Bài 5.** (1) `D = P_a + P_b`, cần `min D[l]`. (2) Các điều kiện như cũ. (3) Đối chiếu vét cạn.

## 19. Đáp án tham khảo cho bài tập

In [20]:
# Bài 1, 2
a1 = Solution().maxDifference("0011100", 7)
a2 = Solution().maxDifference("01110", 2)
assert a1 == max_diff_brute("0011100", 7) == -1 and a2 == max_diff_brute("01110", 2) == 1
print('Bài 1:', a1, '| Bài 2:', a2)

# Bài 3
max_diff_allow_zero = make_solver(require_b_present=False)

def brute_allow_zero(s, k):
    n, best = len(s), NEG
    for i in range(n):
        cnt = [0] * 5
        for j in range(i, n):
            cnt[ord(s[j]) - 48] += 1
            if j - i + 1 >= k:
                for a in range(5):
                    for b in range(5):
                        if a != b and cnt[a] % 2 == 1 and cnt[b] % 2 == 0:
                            best = max(best, cnt[a] - cnt[b])
    return best

rng = random.Random(3)
for _ in range(500):
    s = ''.join(rng.choice('012') for _ in range(rng.randint(3, 12))); k = rng.randint(1, len(s))
    assert max_diff_allow_zero(s, k) == brute_allow_zero(s, k)
print('Bài 3 khớp vét cạn.')

Bài 1: -1 | Bài 2: 1
Bài 3 khớp vét cạn.


In [21]:
# Bài 4, 5
def best_substring(s, k):
    n, ans, where = len(s), NEG, None
    for a in '01234':
        for b in '01234':
            if a == b:
                continue
            Pa, Pb = prefix(s, a), prefix(s, b)
            best, arg = [INF] * 4, [None] * 4
            l = 0
            for r in range(k, n + 1):
                while l <= r - k and Pb[l] < Pb[r]:
                    st = (Pa[l] & 1) << 1 | (Pb[l] & 1)
                    if Pa[l] - Pb[l] < best[st]:
                        best[st], arg[st] = Pa[l] - Pb[l], l
                    l += 1
                need = ((Pa[r] & 1) ^ 1) << 1 | (Pb[r] & 1)
                if best[need] < INF and Pa[r] - Pb[r] - best[need] > ans:
                    ans, where = Pa[r] - Pb[r] - best[need], (arg[need], r, a, b)
    return ans, where

def max_sum_odd_even(s, k):
    n, ans = len(s), NEG
    for a in '01234':
        for b in '01234':
            if a == b:
                continue
            Pa, Pb = prefix(s, a), prefix(s, b)
            best = [INF] * 4
            l = 0
            for r in range(k, n + 1):
                while l <= r - k and Pb[l] < Pb[r]:
                    st = (Pa[l] & 1) << 1 | (Pb[l] & 1)
                    best[st] = min(best[st], Pa[l] + Pb[l])
                    l += 1
                need = ((Pa[r] & 1) ^ 1) << 1 | (Pb[r] & 1)
                if best[need] < INF:
                    ans = max(ans, Pa[r] + Pb[r] - best[need])
    return ans

def brute_sum(s, k):
    n, best = len(s), NEG
    for i in range(n):
        cnt = [0] * 5
        for j in range(i, n):
            cnt[ord(s[j]) - 48] += 1
            if j - i + 1 >= k:
                for a in range(5):
                    for b in range(5):
                        if a != b and cnt[a] % 2 == 1 and cnt[b] and cnt[b] % 2 == 0:
                            best = max(best, cnt[a] + cnt[b])
    return best

rng = random.Random(5)
for _ in range(500):
    s = ''.join(rng.choice('012') for _ in range(rng.randint(3, 12))); k = rng.randint(1, len(s))
    if max_diff_brute(s, k) == NEG:
        continue
    v, (l, r, a, b) = best_substring(s, k)
    sub = s[l:r]
    assert v == max_diff_brute(s, k) and len(sub) >= k and sub.count(a) % 2 == 1
    assert sub.count(b) > 0 and sub.count(b) % 2 == 0 and sub.count(a) - sub.count(b) == v
    assert max_sum_odd_even(s, k) == brute_sum(s, k)
print('Bài 4, 5 khớp vét cạn. Ví dụ 2:', best_substring("1122211", 3))

Bài 4, 5 khớp vét cạn. Ví dụ 2: (1, (0, 5, '2', '1'))


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
for a != b trong 5 ký tự:
    Pa, Pb = tiền tố
    best = [INF]*4; l = 0
    for r in range(k, n+1):
        while l <= r-k and Pb[l] < Pb[r]:
            best[(Pa[l]&1)<<1 | (Pb[l]&1)] = min(..., Pa[l]-Pb[l]); l += 1
        need = ((Pa[r]&1)^1)<<1 | (Pb[r]&1)
        ans = max(ans, Pa[r]-Pb[r]-best[need])
```

**Checklist:**

- [ ] `freq[b]` khác 0 (`Pb[l] < Pb[r]`)?
- [ ] `a` lẻ ⇒ trạng thái tiền tố **khác** tính chẵn lẻ?
- [ ] Độ dài `r − l ≥ k`?
- [ ] Đáp án khởi tạo `−∞` (có thể âm)?
- [ ] Con trỏ `l` chỉ tiến (không reset theo `r`)?